In [1]:
!git clone -b ayush/oceanEmbed-prototype-V2 https://github.com/furged/oceanEmbed-Prototype.git
%cd /content/oceanEmbed-Prototype
!git status

Cloning into 'oceanEmbed-Prototype'...
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 19 (delta 6), reused 4 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (19/19), 1.49 MiB | 9.72 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/oceanEmbed-Prototype
On branch ayush/oceanEmbed-prototype-V2
Your branch is up to date with 'origin/ayush/oceanEmbed-prototype-V2'.

nothing to commit, working tree clean


In [2]:
!pip -q install copernicusmarine xarray netCDF4 zarr dask

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.5/130.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 7.2 MB/s eta 0:00:00


In [3]:
import os
import numpy as np
import xarray as xr
import copernicusmarine
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader

print("imports successful")

imports successful


In [4]:
print("torch:", torch.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

torch: 2.11.0+cu128
gpu: Tesla T4


In [5]:
MIN_LON, MAX_LON = 70, 90
MIN_LAT, MAX_LAT = 5, 20

START_DATE = "2020-01-01"
END_DATE = "2020-06-30"

In [6]:
GLORYS_DATASET = "cmems_mod_glo_phy_my_0.083deg_P1D-m"

GLORYS_VARIABLES = [
    "thetao",
    "so",
    "zos",
    "uo",
    "vo"
]

In [7]:
os.makedirs("data", exist_ok=True)

copernicusmarine.subset(
    dataset_id=GLORYS_DATASET,
    variables=GLORYS_VARIABLES,
    minimum_longitude=MIN_LON,
    maximum_longitude=MAX_LON,
    minimum_latitude=MIN_LAT,
    maximum_latitude=MAX_LAT,
    minimum_depth=0.49,
    maximum_depth=222.48,
    start_datetime=START_DATE,
    end_datetime=END_DATE,
    output_directory="data",
    output_filename="glorys_nio_v2.nc"
)

INFO - 2026-09-05T11:58:01Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register
INFO:copernicusmarine:Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register


Copernicus Marine username: ashakya12
Copernicus Marine password: ··········


INFO - 2026-09-05T11:59:23Z - Selected dataset version: "202311"
INFO:copernicusmarine:Selected dataset version: "202311"
INFO - 2026-09-05T11:59:23Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"
WARNING - 2026-09-05T11:59:23Z - Some of your subset selection [0.49, 222.48] for the depth dimension exceed the dataset coordinates [0.49402499198913574, 5727.9169921875]


  0%|          | [00:00<?]

INFO - 2026-09-05T12:28:36Z - Total size of the download: 2.00 GB.
INFO:copernicusmarine:Total size of the download: 2.00 GB.


ResponseSubset(file_path=PosixPath('data/glorys_nio_v2.nc'), output_directory=PosixPath('data'), filename='glorys_nio_v2.nc', file_size=2045.3717213740458, data_transfer_size=95583.41178625954, variables=['thetao', 'so', 'zos', 'uo', 'vo'], coordinates_extent=[GeographicalExtent(minimum=70.0, maximum=90.0, unit='degrees_east', coordinate_id='longitude'), GeographicalExtent(minimum=5.0, maximum=20.0, unit='degrees_north', coordinate_id='latitude'), TimeExtent(minimum='2020-01-01T00:00:00+00:00', maximum='2020-06-30T00:00:00+00:00', unit='iso8601', coordinate_id='time'), GeographicalExtent(minimum=0.49402499198913574, maximum=222.47520446777344, unit='m', coordinate_id='depth')], status='000', message='The request was successful.', file_status='DOWNLOADED', file_names=None)

In [8]:
glorys_path = "data/glorys_nio_v2.nc"

ds = xr.open_dataset(glorys_path)

ds

<xarray.Dataset> Size: 7GB
Dimensions:    (time: 182, depth: 27, latitude: 181, longitude: 241)
Coordinates:
  * time       (time) datetime64[ns] 1kB 2020-01-01 2020-01-02 ... 2020-06-30
  * depth      (depth) float32 108B 0.494 1.541 2.646 ... 155.9 186.1 222.5
  * latitude   (latitude) float32 724B 5.0 5.083 5.167 5.25 ... 19.83 19.92 20.0
  * longitude  (longitude) float32 964B 70.0 70.08 70.17 ... 89.83 89.92 90.0
Data variables:
    thetao     (time, depth, latitude, longitude) float64 2GB ...
    so         (time, depth, latitude, longitude) float64 2GB ...
    zos        (time, latitude, longitude) float64 64MB ...
    uo         (time, depth, latitude, longitude) float64 2GB ...
    vo         (time, depth, latitude, longitude) float64 2GB ...
Attributes: (12/25)
    Conventions:               CF-1.4
    bulletin_date:             2021-07-07 00:00:00
    bulletin_type:             operational
    comment:                   CMEMS product
    domain_name:               GL12
    easting:                   longitude
    ...                        ...
    references:                http://www.mercator-ocean.fr
    source:                    MERCATOR GLORYS12V1
    title:                     daily mean fields from Global Ocean Physics An...
    z_max:                     5727.9169921875
    z_min:                     0.49402499198913574
    copernicusmarine_version:  2.4.1

In [16]:
!pip -q install cdsapi

In [17]:
from google.colab import userdata
import cdsapi

cds = cdsapi.Client(
    url="https://cds.climate.copernicus.eu/api",
    key=userdata.get("CDS_API_KEY")
)

In [18]:
ERA5_VARIABLES = [
    "10m_u_component_of_wind",
    "10m_v_component_of_wind"
]

ERA5_DAYS = [f"{day:02d}" for day in range(1, 32)]
ERA5_HOURS = [f"{hour:02d}:00" for hour in range(24)]

In [19]:
os.makedirs("data/era5", exist_ok=True)

cds.retrieve(
    "reanalysis-era5-single-levels",
    {
        "product_type": "reanalysis",
        "variable": ERA5_VARIABLES,
        "year": "2020",
        "month": "01",
        "day": ERA5_DAYS,
        "time": ERA5_HOURS,
        "area": [20, 70, 5, 90],
        "data_format": "netcdf",
        "download_format": "unarchived"
    },
    "data/era5/era5_wind_2020_01.nc"
)

2026-09-05 12:50:58,651 INFO Request ID is 8bb432e5-d069-41bb-96cd-ad3cd044b840
INFO:ecmwf.datastores.legacy_client:Request ID is 8bb432e5-d069-41bb-96cd-ad3cd044b840
2026-09-05 12:50:58,813 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-09-05 12:51:12,589 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-09-05 12:52:20,853 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


4d4cef443c8cf0ca6bb24aa8695aa226.nc:   0%|          | 0.00/15.2M [00:00<?, ?B/s]

'data/era5/era5_wind_2020_01.nc'

In [20]:
for month in range(2, 7):
    month_str = f"{month:02d}"

    cds.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": ERA5_VARIABLES,
            "year": "2020",
            "month": month_str,
            "day": ERA5_DAYS,
            "time": ERA5_HOURS,
            "area": [20, 70, 5, 90],
            "data_format": "netcdf",
            "download_format": "unarchived"
        },
        f"data/era5/era5_wind2020{month_str}.nc"
    )

2026-09-05 12:57:34,219 INFO Request ID is 9f1761c2-4689-4aa4-a7a9-84401d3e83ff
INFO:ecmwf.datastores.legacy_client:Request ID is 9f1761c2-4689-4aa4-a7a9-84401d3e83ff
2026-09-05 12:57:34,362 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-09-05 12:57:49,309 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-09-05 13:01:56,874 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


eb04f095a5f0314eaa327ca1be2ac6b.nc:   0%|          | 0.00/14.3M [00:00<?, ?B/s]

2026-09-05 13:01:59,952 INFO Request ID is 4f8abefe-29be-4c33-a26a-c5b2167e8a63
INFO:ecmwf.datastores.legacy_client:Request ID is 4f8abefe-29be-4c33-a26a-c5b2167e8a63
2026-09-05 13:02:00,111 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-09-05 13:02:14,952 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-09-05 13:03:18,508 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


aabd6a223d2ee25b40c2e3b6ac343ce5.nc:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

2026-09-05 13:03:24,832 INFO Request ID is dd60131d-b029-442e-a85e-5d435cb5b810
INFO:ecmwf.datastores.legacy_client:Request ID is dd60131d-b029-442e-a85e-5d435cb5b810
2026-09-05 13:03:24,996 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-09-05 13:03:38,917 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-09-05 13:04:41,172 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


3ce969338763fe5cc5402af9da2589b1.nc:   0%|          | 0.00/15.0M [00:00<?, ?B/s]

2026-09-05 13:04:44,784 INFO Request ID is 1b515a8c-4ce2-4050-95b5-6f384e02531c
INFO:ecmwf.datastores.legacy_client:Request ID is 1b515a8c-4ce2-4050-95b5-6f384e02531c
2026-09-05 13:04:44,917 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-09-05 13:05:06,618 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-09-05 13:06:01,285 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


337332de6d3922ae2d429c9718356b88.nc:   0%|          | 0.00/15.7M [00:00<?, ?B/s]

2026-09-05 13:06:04,817 INFO Request ID is 0136e417-27a2-41fa-b47a-bae35cf66b25
INFO:ecmwf.datastores.legacy_client:Request ID is 0136e417-27a2-41fa-b47a-bae35cf66b25
2026-09-05 13:06:04,968 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-09-05 13:06:15,508 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-09-05 13:07:25,033 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


fedb8b523f31347c676a8c804e15ea71.nc:   0%|          | 0.00/14.9M [00:00<?, ?B/s]

In [23]:
import os

os.listdir("data/era5")

['era5_wind202004.nc',
 'era5_wind_2020_01.nc',
 'era5_wind202006.nc',
 'era5_wind202002.nc',
 'era5_wind202003.nc',
 'era5_wind202005.nc']

In [25]:
era5_datasets = [xr.open_dataset(f) for f in era5_files]

era5 = xr.concat(
    era5_datasets,
    dim="valid_time",
    join="exact"
).rename({"valid_time": "time"})

era5

<xarray.Dataset> Size: 173MB
Dimensions:    (time: 4368, latitude: 61, longitude: 81)
Coordinates:
  * time       (time) datetime64[ns] 35kB 2020-01-01 ... 2020-06-30T23:00:00
  * latitude   (latitude) float64 488B 20.0 19.75 19.5 19.25 ... 5.5 5.25 5.0
  * longitude  (longitude) float64 648B 70.0 70.25 70.5 ... 89.5 89.75 90.0
    number     int64 8B 0
    expver     (time) <U4 70kB '0001' '0001' '0001' ... '0001' '0001' '0001'
Data variables:
    u10        (time, latitude, longitude) float32 86MB -2.526 -2.275 ... 2.85
    v10        (time, latitude, longitude) float32 86MB -3.35 -2.675 ... 3.403
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-09-05T12:51 GRIB to CDM+CF via cfgrib-0.9.1...

In [26]:
era5.latitude.values[:5], era5.longitude.values[:5]

(array([20.  , 19.75, 19.5 , 19.25, 19.  ]),
 array([70.  , 70.25, 70.5 , 70.75, 71.  ]))

In [27]:
era5_regridded = era5.interp(
    latitude=ds.latitude,
    longitude=ds.longitude,
    method="linear"
)

era5_regridded

<xarray.Dataset> Size: 3GB
Dimensions:    (time: 4368, latitude: 181, longitude: 241)
Coordinates:
  * time       (time) datetime64[ns] 35kB 2020-01-01 ... 2020-06-30T23:00:00
  * latitude   (latitude) float32 724B 5.0 5.083 5.167 5.25 ... 19.83 19.92 20.0
  * longitude  (longitude) float32 964B 70.0 70.08 70.17 ... 89.83 89.92 90.0
    number     int64 8B 0
    expver     (time) <U4 70kB '0001' '0001' '0001' ... '0001' '0001' '0001'
Data variables:
    u10        (time, latitude, longitude) float64 2GB -5.539 -5.534 ... 3.655
    v10        (time, latitude, longitude) float64 2GB 0.3233 0.2972 ... 2.905
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-09-05T12:51 GRIB to CDM+CF via cfgrib-0.9.1...

In [28]:
era5_daily = era5_regridded.resample(time="1D").mean()

era5_daily

<xarray.Dataset> Size: 127MB
Dimensions:    (time: 182, latitude: 181, longitude: 241)
Coordinates:
  * time       (time) datetime64[ns] 1kB 2020-01-01 2020-01-02 ... 2020-06-30
  * latitude   (latitude) float32 724B 5.0 5.083 5.167 5.25 ... 19.83 19.92 20.0
  * longitude  (longitude) float32 964B 70.0 70.08 70.17 ... 89.83 89.92 90.0
    number     int64 8B 0
Data variables:
    u10        (time, latitude, longitude) float64 64MB -5.697 -5.694 ... 3.016
    v10        (time, latitude, longitude) float64 64MB 0.07145 0.05863 ... 3.83
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-09-05T12:51 GRIB to CDM+CF via cfgrib-0.9.1...

In [29]:
ds.time.equals(era5_daily.time)

False

In [30]:
print(ds.time.values[:3])
print(era5_daily.time.values[:3])

['2020-01-01T00:00:00.000000000' '2020-01-02T00:00:00.000000000'
 '2020-01-03T00:00:00.000000000']
['2020-01-01T00:00:00.000000000' '2020-01-02T00:00:00.000000000'
 '2020-01-03T00:00:00.000000000']


In [31]:
era5_daily = era5_daily.sel(time=ds.time)

era5_daily.time.equals(ds.time)

False

In [32]:
print(ds.time.dtype)
print(era5_daily.time.dtype)
print((ds.time.values - era5_daily.time.values)[:5])

datetime64[ns]
datetime64[ns]
[0 0 0 0 0]


In [33]:
era5_daily = era5_daily.assign_coords(time=ds.time)

era5_daily.time.equals(ds.time)

False

In [34]:
era5_daily = era5_daily.assign_coords(time=ds.time)

print(era5_daily.dims)


FrozenMappingWarningOnValuesAccess({'time': 182, 'latitude': 181, 'longitude': 241})


In [36]:
sst = ds.thetao.isel(depth=0, drop=True)
sss = ds.so.isel(depth=0, drop=True)
ssh = ds.zos

current_u = ds.uo.isel(depth=0, drop=True)
current_v = ds.vo.isel(depth=0, drop=True)

wind_u = era5_daily.u10
wind_v = era5_daily.v10

X = xr.concat(
    [sst, sss, ssh, current_u, current_v, wind_u, wind_v],
    dim="channel",
    coords="minimal",
    compat="override"
)

X = X.transpose("time", "channel", "latitude", "longitude")

X.shape

(182, 7, 181, 241)

In [37]:
target_depths = [
    0.494025, 1.541375, 2.645669, 5.078224, 7.92956,
    11.405, 15.81007, 21.59882, 29.44473, 40.34405,
    55.76429, 77.85385, 109.7293, 155.8507, 222.4752
]

Y = ds.thetao.sel(depth=target_depths, method="nearest")

Y = Y.transpose("time", "depth", "latitude", "longitude")

Y.shape

(182, 15, 181, 241)

In [38]:
n_time = X.sizes["time"]

train_end = int(0.70 * n_time)
val_end = int(0.85 * n_time)

X_train = X.isel(time=slice(0, train_end))
X_val = X.isel(time=slice(train_end, val_end))
X_test = X.isel(time=slice(val_end, None))

Y_train = Y.isel(time=slice(0, train_end))
Y_val = Y.isel(time=slice(train_end, val_end))
Y_test = Y.isel(time=slice(val_end, None))

X_train.shape, X_val.shape, X_test.shape

((127, 7, 181, 241), (27, 7, 181, 241), (28, 7, 181, 241))

In [40]:
x_mean = X_train.mean(dim=("time", "latitude", "longitude"))
x_std = X_train.std(dim=("time", "latitude", "longitude"))

y_mean = Y_train.mean(dim=("time", "latitude", "longitude"))
y_std = Y_train.std(dim=("time", "latitude", "longitude"))

X_train = (X_train - x_mean) / x_std
X_val = (X_val - x_mean) / x_std
X_test = (X_test - x_mean) / x_std

Y_train = (Y_train - y_mean) / y_std
Y_val = (Y_val - y_mean) / y_std
Y_test = (Y_test - y_mean) / y_std

In [41]:
ocean_mask = np.isfinite(X_train.isel(time=0).isel(channel=0).values)

ocean_mask.shape, ocean_mask.sum()


((181, 241), np.int64(31800))

In [42]:
X_train = X_train.fillna(0)
X_val = X_val.fillna(0)
X_test = X_test.fillna(0)

Y_train = Y_train.fillna(0)
Y_val = Y_val.fillna(0)
Y_test = Y_test.fillna(0)

In [43]:
X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
X_val_t = torch.tensor(X_val.values, dtype=torch.float32)
X_test_t = torch.tensor(X_test.values, dtype=torch.float32)

Y_train_t = torch.tensor(Y_train.values, dtype=torch.float32)
Y_val_t = torch.tensor(Y_val.values, dtype=torch.float32)
Y_test_t = torch.tensor(Y_test.values, dtype=torch.float32)

In [44]:
batch_size = 2

train_loader = DataLoader(
    TensorDataset(X_train_t, Y_train_t),
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val_t, Y_val_t),
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(X_test_t, Y_test_t),
    batch_size=batch_size,
    shuffle=False
)

In [45]:
mask_t = torch.tensor(
    ocean_mask,
    dtype=torch.bool
)

mask_t.shape

torch.Size([181, 241])

In [46]:
class SpectralConv3d(nn.Module):
    def init(self, in_channels, out_channels, modes_t, modes_x, modes_y):
        super().init()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes_t = modes_t
        self.modes_x = modes_x
        self.modes_y = modes_y

        scale = 1 / (in_channels * out_channels)

        self.weights = nn.Parameter(
            scale * torch.rand(
                in_channels,
                out_channels,
                modes_t,
                modes_x,
                modes_y,
                dtype=torch.cfloat
            )
        )

    def forward(self, x):
        x_ft = torch.fft.rfftn(
            x,
            dim=(-3, -2, -1)
        )

        out_ft = torch.zeros(
            x.size(0),
            self.out_channels,
            x.size(-3),
            x.size(-2),
            x_ft.size(-1),
            dtype=torch.cfloat,
            device=x.device
        )

        out_ft[
            :, :, :self.modes_t,
            :self.modes_x,
            :self.modes_y
        ] = torch.einsum(
            "bixyz,ioxyz->boxyz",
            x_ft[
                :, :, :self.modes_t,
                :self.modes_x,
                :self.modes_y
            ],
            self.weights
        )

        return torch.fft.irfftn(
            out_ft,
            s=x.shape[-3:]
        )

In [47]:
class HybridFNOBlock3d(nn.Module):
    def init(self, channels, modes_t, modes_x, modes_y):
        super().init()

        self.spectral = SpectralConv3d(
            channels,
            channels,
            modes_t,
            modes_x,
            modes_y
        )

        self.local = nn.Conv3d(
            channels,
            channels,
            kernel_size=1
        )

        self.activation = nn.GELU()

    def forward(self, x):
        x = self.spectral(x) + self.local(x)
        return self.activation(x)

In [48]:
class OceanEmbed3D(nn.Module):
    def init(self):
        super().init()

        self.lift = nn.Conv3d(
            7, 32,
            kernel_size=1
        )

        self.block1 = HybridFNOBlock3d(
            32, 8, 16, 16
        )

        self.block2 = HybridFNOBlock3d(
            32, 8, 16, 16
        )

        self.block3 = HybridFNOBlock3d(
            32, 8, 16, 16
        )

        self.project = nn.Sequential(
            nn.Conv3d(32, 64, kernel_size=1),
            nn.GELU(),
            nn.Conv3d(64, 15, kernel_size=1)
        )

    def forward(self, x):
        x = self.lift(x)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)

        return self.project(x)

In [49]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = OceanEmbed3D().to(device)

sum(p.numel() for p in model.parameters() if p.requires_grad)

0

In [50]:
print(type(model))
print(len(list(model.parameters())))
print(sum(p.numel() for p in model.parameters()))

<class '__main__.OceanEmbed3D'>
0
0


In [54]:
class OceanEmbed3D(nn.Module):
    def init(self):
        super().init()

        self.lift = nn.Conv3d(7, 32, 1)

        self.block1 = HybridFNOBlock3d(32, 8, 16, 16)
        self.block2 = HybridFNOBlock3d(32, 8, 16, 16)
        self.block3 = HybridFNOBlock3d(32, 8, 16, 16)

        self.project1 = nn.Conv3d(32, 64, 1)
        self.project2 = nn.Conv3d(64, 15, 1)

    def forward(self, x):
        x = self.lift(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = torch.nn.functional.gelu(self.project1(x))
        x = self.project2(x)
        return x


model = OceanEmbed3D().to(device)

print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))

OceanEmbed3D()
parameters: 0


In [59]:
class TestModel(nn.Module):
    def init(self):
        super().init()
        self.layer = nn.Conv3d(7, 32, 1)

test_model = TestModel()

print(test_model)
print("parameters:", sum(p.numel() for p in test_model.parameters()))

TestModel()
parameters: 0


In [64]:
!git config --global user.email "ayushmaandhiman59@gmail.com"
!git config --global user.name "Ayushu69"

In [65]:
!git add notebooks/
!git add src/
!git add README.md
!git commit -m "update v2 training pipeline"
!git push origin ayush/oceanEmbed-prototype-V2

fatal: pathspec 'notebooks/' did not match any files
fatal: pathspec 'src/' did not match any files
On branch ayush/oceanEmbed-prototype-V2
Your branch is up to date with 'origin/ayush/oceanEmbed-prototype-V2'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/

nothing added to commit but untracked files present (use "git add" to track)
fatal: could not read Username for 'https://github.com': No such device or address


In [66]:
!pwd
!ls
!git remote -v

/content/oceanEmbed-Prototype
data  README.md
origin	https://github.com/furged/oceanEmbed-Prototype.git (fetch)
origin	https://github.com/furged/oceanEmbed-Prototype.git (push)


In [67]:
!find /content -name "*.ipynb"

In [68]:
!ls -la /content/oceanEmbed-Prototype

total 20
drwxr-xr-x 4 root root 4096 Sep  5 11:58 .
drwxr-xr-x 1 root root 4096 Sep  5 11:57 ..
drwxr-xr-x 3 root root 4096 Sep  5 12:35 data
drwxr-xr-x 8 root root 4096 Sep  5 14:31 .git
-rw-r--r-- 1 root root  166 Sep  5 11:57 README.md


In [69]:
!cp /content/*.ipynb /content/oceanEmbed-Prototype/

cp: cannot stat '/content/*.ipynb': No such file or directory
